# ARC AGI 3 Colab Training

This notebook is set up for the stable Drive based workflow.

1. Read the latest project code from `MyDrive/ARC Prize 2026 - ARC-AGI-3`
2. Copy that code to the Colab local disk
3. Read one or more training `.gz` files from `MyDrive/ARC2026_AGI_3/Input_Data`
4. Train on Colab local disk
5. Write checkpoints and metrics to `MyDrive/ARC2026_AGI_3/Training_Output/<timestamp>/`

Keep the notebook thin. Keep the real project code in files.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime

PROJECT_SOURCE_MODE = 'drive_dir'  # Default and recommended. Use 'drive_zip' only if you uploaded a project bundle zip.
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/ARC Prize 2026 - ARC-AGI-3')
DRIVE_PROJECT_ZIP = Path('/content/drive/MyDrive/ARC Prize 2026 - ARC-AGI-3/Local_Output/Colab_Bundles/arc_agi3_colab_bundle.zip')
DRIVE_INPUT_DATA_BASE = Path('/content/drive/MyDrive/ARC2026_AGI_3/Input_Data')
DRIVE_OUTPUT_BASE = Path('/content/drive/MyDrive/ARC2026_AGI_3/Training_Output')
LOCAL_WORKDIR = Path('/content/ARC Prize 2026 - ARC-AGI-3')
RUN_TS = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
OUTPUT_ROOT = DRIVE_OUTPUT_BASE / RUN_TS
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_INPUT_DATA_BASE.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

print('Project source mode:', PROJECT_SOURCE_MODE)
print('Drive project root:', DRIVE_PROJECT_ROOT)
print('Drive project zip:', DRIVE_PROJECT_ZIP)
print('Drive input data base:', DRIVE_INPUT_DATA_BASE)
print('Drive output base:', DRIVE_OUTPUT_BASE)
print('Local workdir:', LOCAL_WORKDIR)
print('Output root:', OUTPUT_ROOT)


In [ ]:
import shutil

if LOCAL_WORKDIR.exists():
    shutil.rmtree(LOCAL_WORKDIR)
LOCAL_WORKDIR.parent.mkdir(parents=True, exist_ok=True)

if PROJECT_SOURCE_MODE == 'drive_dir':
    get_ipython().system('rsync -a --delete --exclude .git "{}"/ "{}"/'.format(DRIVE_PROJECT_ROOT, LOCAL_WORKDIR))
elif PROJECT_SOURCE_MODE == 'drive_zip':
    if not DRIVE_PROJECT_ZIP.exists():
        raise FileNotFoundError(f'Project zip not found: {DRIVE_PROJECT_ZIP}')
    get_ipython().system('unzip -q "{}" -d /content'.format(DRIVE_PROJECT_ZIP))
else:
    raise ValueError(f'Unsupported PROJECT_SOURCE_MODE: {PROJECT_SOURCE_MODE}')

get_ipython().run_line_magic('cd', str(LOCAL_WORKDIR))


In [ ]:
import os, sys, subprocess

def run(cmd):
    print('>>>', cmd)
    subprocess.check_call(cmd, shell=True)

run('python -m pip install -U pip wheel setuptools')
run('python -m pip install -U torch torchvision torchaudio')

try:
    run('python -m pip install -U arc-agi==0.9.8 arcengine==0.9.3')
except Exception:
    print('PyPI install failed, trying local wheels...')
    run('python -m pip install arc_agi_3_wheels/*.whl')

run('python - <<\'PY\'\nimport torch\nprint("torch", torch.__version__)\nprint("cuda", torch.cuda.is_available())\nif torch.cuda.is_available():\n    print("device", torch.cuda.get_device_name(0))\nPY')


In [ ]:
HARDWARE_PROFILE = 'a100'  # Training profile.
COLLECT_PROFILE = 'a100'
COLLECT_TAG = 'public_search_a100_v1'
RUN_COLLECTION = False  # Usually leave this off for Colab training runs.
RUN_HUMAN_IMPORT = False  # Turn this on only if you uploaded the raw human zip and want Colab to convert it.
HUMAN_ZIP_PATH = DRIVE_INPUT_DATA_BASE / 'arc_agi_3_public_demo_human_testing.zip'
HUMAN_IMPORT_OUTPUT = DRIVE_INPUT_DATA_BASE / 'arc_agi_3_public_demo_human_testing.gz'
HUMAN_IMPORT_GAMES = None  # None means all 25 public games.
HUMAN_IMPORT_MIN_LEVELS = 0
HUMAN_IMPORT_TOP_K = None
COLLECT_STEPS = 96
COLLECT_WORKERS = 8
COLLECT_GAMES = None
COLLECT_EPISODES_PER_GAME = None
COLLECT_BEAM_WIDTH = None
COLLECT_BRANCH_FACTOR = None
CHECKPOINT_EVERY_STEPS = 100
RESUME_CHECKPOINT = None

COLLECT_ROOT = DRIVE_INPUT_DATA_BASE / COLLECT_TAG
TRAIN_DATA_PATHS = [DRIVE_INPUT_DATA_BASE / 'arc_agi_3_public_demo_human_testing.gz']
collect_optional_args = []
collect_optional_args.append(f'--workers {COLLECT_WORKERS}')
if COLLECT_GAMES:
    collect_optional_args.append(f'--games {COLLECT_GAMES}')
if COLLECT_EPISODES_PER_GAME is not None:
    collect_optional_args.append(f'--episodes-per-game {COLLECT_EPISODES_PER_GAME}')
if COLLECT_BEAM_WIDTH is not None:
    collect_optional_args.append(f'--beam-width {COLLECT_BEAM_WIDTH}')
if COLLECT_BRANCH_FACTOR is not None:
    collect_optional_args.append(f'--branch-factor {COLLECT_BRANCH_FACTOR}')
collect_optional = '' if not collect_optional_args else ' \
  ' + ' \
  '.join(collect_optional_args)

if RUN_HUMAN_IMPORT:
    human_parts = [
        'python -m src.import_human_replays',
        f'  --project-root "{LOCAL_WORKDIR}"',
        f'  --input "{HUMAN_ZIP_PATH}"',
        f'  --output "{HUMAN_IMPORT_OUTPUT}"',
        f'  --min-levels {HUMAN_IMPORT_MIN_LEVELS}',
    ]
    if HUMAN_IMPORT_GAMES:
        human_parts.append(f'  --games "{HUMAN_IMPORT_GAMES}"')
    if HUMAN_IMPORT_TOP_K is not None:
        human_parts.append(f'  --top-k-per-game {HUMAN_IMPORT_TOP_K}')
    human_cmd = ' \
'.join(human_parts)
    run(human_cmd)
    if HUMAN_IMPORT_OUTPUT not in TRAIN_DATA_PATHS:
        TRAIN_DATA_PATHS.insert(0, HUMAN_IMPORT_OUTPUT)

if RUN_COLLECTION:
    collect_cmd = f'''python -m src.collect \
  --project-root "{LOCAL_WORKDIR}" \
  --output-root "{COLLECT_ROOT}" \
  --hardware-profile {COLLECT_PROFILE} \
  --seeds 0,1,2,3 \
  --max-steps {COLLECT_STEPS}{collect_optional}'''
    run(collect_cmd)
    TRAIN_DATA_PATHS.append(COLLECT_ROOT / 'collected' / 'episodes.jsonl.gz')

TRAIN_DATA_PATHS = [Path(path) for path in TRAIN_DATA_PATHS]
TRAIN_DATA_PATHS = [path for path in TRAIN_DATA_PATHS if path.exists()]
if not TRAIN_DATA_PATHS:
    raise FileNotFoundError('No training data paths were found under ARC2026_AGI_3/Input_Data.')

TRAIN_DATA_ARG = ','.join(str(path) for path in TRAIN_DATA_PATHS)
print('Training data paths:')
for path in TRAIN_DATA_PATHS:
    print(' -', path)


In [ ]:
resume_arg = '' if RESUME_CHECKPOINT is None else f' \\\n  --resume "{RESUME_CHECKPOINT}"'

train_cmd = f'''python -m src.train \
  --project-root "{LOCAL_WORKDIR}" \
  --data "{TRAIN_DATA_ARG}" \
  --output-dir "{OUTPUT_ROOT}" \
  --hardware-profile {HARDWARE_PROFILE} \
  --max-steps 192 \
  --online-val-games 5 \
  --checkpoint-every-steps {CHECKPOINT_EVERY_STEPS}{resume_arg}'''

run(train_cmd)


In [ ]:
eval_cmd = f'''python -m src.evaluate \
  --project-root "{LOCAL_WORKDIR}" \
  --checkpoint "{OUTPUT_ROOT / 'checkpoints' / 'best.pth'}" \
  --output "{OUTPUT_ROOT / 'public_eval.json'}" \
  --split val'''

run(eval_cmd)


In [ ]:
import pandas as pd
from pathlib import Path

metrics_path = OUTPUT_ROOT / 'metrics.csv'
display(pd.read_csv(metrics_path).tail())
print('Best checkpoint:', OUTPUT_ROOT / 'checkpoints' / 'best.pth')
print('Public eval:', OUTPUT_ROOT / 'public_eval.json')
